# PONITA + ECT on rMD17 with PONITA's official training setup (resumable)

Same models as `colab_rmd17.ipynb`, trained with `--official`, which matches PONITA's own MD17 code
(`main_md17.py` + `lightning_wrappers/md17.py`):
- squared-error loss, λ_F = 500
- **energy = sum of per-atom energies** (no MLP head)
- **random rotation** of every training batch
- **EMA of the weights (0.99)**: validation, checkpoints and test use the averaged weights
- **self-loops** in the graph
- targets scaled by the **force RMS**
- 100-epoch warmup, as in their code

**Run the baseline (Cell 6) first, on its own.** Only if it lands near PONITA's published aspirin result
(E 1.7 meV, F 5.8 meV/Å) is it worth running the ECT models (Cells 7–11) with this setup.

Checkpoints go to `dino_ckpts_official` on Drive, so earlier runs are untouched. If Colab disconnects, run Cells 1–3 again,
then the cell that was running. It resumes from the last checkpoint.
Run cells one at a time, not "Run all". Cell 5 (data) only needs to work once, since the rMD17 file is reused from your earlier setup.

**Cell 11b:** regularised concat (16-dim token, dropout 0.3 on the token). In the curves and the table it appears as `concat-reg`.


### One-time setup
The repo on Drive (`MyDrive/dino`) must have the latest `train_ponita_md17.py`, `models/ponita_ect.py`,
`models/egnn_ect.py` and `qm9_ect.py`.

In [ ]:
# Cell 1 — confirm you have a GPU
!nvidia-smi

In [ ]:
# Cell 2 — mount Drive, set paths and shared settings (re-run after any reconnect)
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/dino'
CKPT_DIR = '/content/drive/MyDrive/dino_ckpts_official'  # new folder: earlier runs stay untouched
DATA_DIR = '/content/drive/MyDrive/dino_data/md17'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
assert os.path.exists(os.path.join(REPO_DIR, 'train_ponita_md17.py')), \
    f'train_ponita_md17.py not found in {REPO_DIR} — fix REPO_DIR'

EPOCHS   = 5000
SEED     = 0
MOLECULE = 'revised-aspirin'   # rMD17, as in the PONITA paper
LOSS     = 'mse'               # set by --official
MOL      = 'revised_aspirin'   # how the molecule appears in checkpoint filenames
SUFFIX   = '_mse_sum_aug_ema099_loop_frms'   # how --official runs are named on disk
COMMON = (f'--molecule {MOLECULE} --official --epochs {EPOCHS} --seed {SEED} '
          f'--data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" '
          '--log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 '
          '--lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 '
          '--basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 --icosphere-nu 3')
os.chdir(REPO_DIR)
print('Repo :', REPO_DIR)
print('Ckpts:', CKPT_DIR)
print('Data :', DATA_DIR)

In [ ]:
# Cell 3 — install dependencies matching Colab's torch build
# (ponita needs Python 3.10+, which Colab has — so NONE of the cluster's
#  match/case patches are needed here.)
import torch
v = torch.__version__.split('+')[0]
cu = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
wheel = f'https://data.pyg.org/whl/torch-{v}+{cu}.html'
print('torch', torch.__version__, '->', wheel)
!pip -q install pyg-lib -f {wheel}
!pip -q install torch-scatter -f {wheel}
!pip -q install torch-geometric
!pip -q install git+https://github.com/ebekkers/ponita.git

In [ ]:
# Cell 4 — sanity check: imports work and the ECT is rotation-invariant
os.chdir(REPO_DIR)
!python test_equivariance.py

In [ ]:
# Cell 5 — get rMD17 aspirin into the folder PyG reads (once; cached on Drive afterwards)
# PyG's own rMD17 link returns 404. Download the release from https://doi.org/10.6084/m9.figshare.12672038,
# then upload the extracted rmd17 folder (or just rmd17_aspirin.npz) anywhere under MyDrive in a folder
# whose name starts with "rmd17". This cell finds the file and copies it to where PyG expects it.
import glob, os, shutil
from torch_geometric.datasets import MD17

expected = os.path.join(DATA_DIR, 'raw', 'rmd17', 'npz_data', 'rmd17_aspirin.npz')   # PyG only looks here

if not os.path.exists(expected):
    found = sorted(glob.glob('/content/drive/MyDrive/rmd17*/**/rmd17_aspirin.npz', recursive=True)
                   + glob.glob('/content/drive/MyDrive/rmd17_aspirin.npz'))
    if not found:
        raise FileNotFoundError('rmd17_aspirin.npz not found under MyDrive/rmd17*. Upload it, then re-run.')
    print('Found  ', found[0])
    os.makedirs(os.path.dirname(expected), exist_ok=True)
    shutil.copy(found[0], expected)
    print('Copied to', expected)

ds = MD17(root=DATA_DIR, name='revised aspirin')
print(f'rMD17 aspirin ready: {len(ds)} frames')


In [ ]:
# Cell 6 — baseline FIRST. Compare with PONITA's published E 1.7 meV / F 5.8 meV/Å before running anything else.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/baseline_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model baseline {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 7 — concat (resumes automatically). Global ECT token concatenated at the input.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/concat_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model ect --cond-type concat {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 8 — film-final (resumes automatically). Global ECT token, FiLM on the final embedding.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/film-final_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model ect --cond-type film-final {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 9 — leap (resumes automatically). Per-atom local ECT concatenated at the input (5 Å).
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/leap_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model ect --cond-type leap {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 10 — hybrid-adaln (resumes automatically). AdaLN in every block from the global ECT, plus global ECT concat before the head.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/hybrid-adaln_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model ect --cond-type hybrid-adaln {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 11 — hybrid-leap (resumes automatically). leap (per-atom local ECT at the input), plus global ECT concat before the head. No AdaLN.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/hybrid-leap_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model ect --cond-type hybrid-leap {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 11b — concat, regularised: 16-dim ECT token + dropout 0.3 on the token (resumes automatically).
# Tests whether concat's overfitting comes from the token acting as a per-geometry fingerprint.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/concat_reg_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model ect --cond-type concat {COMMON} --ect-out-dim 16 --ect-dropout 0.3 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 11c — finite cutoff, baseline (resumes automatically). PONITA's smooth cutoff at RC Å:
# atoms RC or more apart no longer interact directly (aspirin is ~9 Å across, so at 1000 Å all do).
# 3.5 Å removes 57% of aspirin's atom pairs (5 Å only 28%). Run this before Cell 11d: if test F stays
# near 8.3, the cutoff costs nothing on aspirin and there is nothing for the ECT to recover.
RC = 3.5
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/baseline_pr{RC:g}_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model baseline {COMMON} --ponita-radius {RC} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 11d — finite cutoff, concat (resumes automatically). Same cutoff as Cell 11c, plus the global ECT
# token at the input. Does the token give back what the cutoff took away? Compare with Cell 11c, not Cell 6.
RC = 3.5
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/concat_pr{RC:g}_seed{SEED}_ep{EPOCHS}{SUFFIX}.log'
!python -u train_ponita_md17.py --model ect --cond-type concat {COMMON} --ponita-radius {RC} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 12 — training curves for every model that has started (re-run any time)
import glob, json, re
import numpy as np
import matplotlib.pyplot as plt

# Fixed colour per model, so a model keeps its colour whichever runs exist.
STYLE = {
    'baseline':     dict(color='#8a8880', ls='--'),
    'concat':       dict(color='#2a78d6', ls='-'),
    'film-final':   dict(color='#eb6834', ls='-'),
    'leap':         dict(color='#1baf7a', ls='-'),
    'hybrid-adaln': dict(color='#eda100', ls='-'),
    'hybrid-leap':  dict(color='#e87ba4', ls='-'),
    'concat-reg':   dict(color='#2a78d6', ls=':'),
}

def model_name(path):
    base = os.path.basename(path)
    name = 'baseline' if 'PONITA_baseline' in base else re.search(r'ECT_(.+?)_' + MOL + '_seed', base).group(1)
    extra = base.split(f'_seed{SEED}{SUFFIX}')[1].rsplit('_', 1)[0]   # e.g. '_d16_drop03' for the regularised run
    cutoff = re.search(r'_pr([0-9p]+)', extra)                        # e.g. '_pr5' for a finite PONITA cutoff
    if cutoff:
        return f'{name}-pr{cutoff.group(1)}'
    return name + ('-reg' if extra else '')

def style(name):
    # Finite-cutoff runs keep their model's colour, dash-dotted.
    if '-pr' in name:
        return {**STYLE[name.split('-pr')[0]], 'ls': '-.'}
    return STYLE[name]

def smooth(y, k=25):
    # Train MAE is logged every epoch and is noisy; show a 25-epoch moving average.
    y = np.asarray(y, dtype=float)
    if len(y) < k:
        return y
    return np.convolve(y, np.ones(k) / k, mode='valid')

hists = {}
for path in sorted(glob.glob(f'{CKPT_DIR}/*_{MOL}_seed{SEED}{SUFFIX}*_history.json')):
    if '_cdrop' in path or '_cnoise' in path:
        continue   # condition-dropout / noise runs are plotted in colab_cond_reg.ipynb
    name = model_name(path)
    if name.split('-pr')[0] in STYLE:
        with open(path) as f:
            hists[name] = json.load(f)
assert hists, f'No *_history.json in {CKPT_DIR} yet — start a training cell first.'
order = [m for m in STYLE if m in hists] + sorted(m for m in hists if m not in STYLE)

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=True)
panels = [
    (axes[0, 0], 'train_f', 'Train force MAE (meV/Å), 25-epoch average'),
    (axes[0, 1], 'val_f',   'Validation force MAE (meV/Å)'),
    (axes[1, 0], 'train_e', 'Train energy MAE (meV), 25-epoch average'),
    (axes[1, 1], 'val_e',   'Validation energy MAE (meV)'),
]
for ax, key, title in panels:
    for name in order:
        h = hists[name]
        if key.startswith('train'):
            y = smooth(h[key])
            x = np.asarray(h['epoch'][len(h['epoch']) - len(y):])
        else:
            x, y = np.asarray(h['val_epoch']), np.asarray(h[key])
        if len(y):
            ax.plot(x, y, lw=1.8, label=name, **style(name))
    ax.set_title(title, fontsize=11, loc='left')
    ax.set_yscale('log')
    ax.grid(True, which='both', color='#e4e2dc', lw=0.6)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
for ax in axes[1]:
    ax.set_xlabel('epoch')
axes[0, 1].legend(frameon=False, fontsize=9, loc='upper right')
fig.suptitle(f'PONITA + ECT on rMD17 aspirin, PONITA official setup, seed {SEED} ({EPOCHS} epochs)', x=0.01, ha='left', fontsize=13)
fig.tight_layout()
out = f'{CKPT_DIR}/training_curves_seed{SEED}{SUFFIX}.png'
fig.savefig(out, dpi=150)
plt.show()
print('Saved', out)
print('Epochs done:', {m: len(hists[m]['epoch']) for m in order})

In [ ]:
# Cell 13 — results table: test MAE at the best-force and best-energy checkpoints
import glob, json, re
import pandas as pd

rows = []
for path in sorted(glob.glob(f'{CKPT_DIR}/*_{MOL}_seed{SEED}{SUFFIX}*_results.json')):
    with open(path) as f:
        r = json.load(f)
    if r.get('config', {}).get('cond_drop') or r.get('config', {}).get('cond_noise'):
        continue   # condition-dropout / noise runs are in colab_cond_reg.ipynb
    name = 'baseline' if 'baseline' in r['label'] else re.search(r'\((.+)\)', r['label']).group(1)
    if r.get('config', {}).get('ect_dropout') or r.get('config', {}).get('ect_out_dim', 64) != 64:
        name += '-reg'
    radius = r.get('config', {}).get('ponita_radius', 1000.0)
    if radius != 1000.0:
        name += f'-pr{radius:g}'.replace('.', 'p')   # finite PONITA cutoff (Cells 11c, 11d)
    for which, ckpt in (('best_f', 'best val F'), ('best_e', 'best val E')):
        if which in r:
            rows.append({'model': name, 'checkpoint': ckpt,
                         'E (meV)': r[which]['test_e'], 'F (meV/Å)': r[which]['test_f']})
assert rows, 'No *_results.json yet — they are written when a model finishes training.'

df = pd.DataFrame(rows)
base = df[df.model == 'baseline'].set_index('checkpoint')
if len(base):
    df['Δ E'] = df.apply(lambda r: r['E (meV)'] / base.loc[r.checkpoint, 'E (meV)'] - 1, axis=1)
    df['Δ F'] = df.apply(lambda r: r['F (meV/Å)'] / base.loc[r.checkpoint, 'F (meV/Å)'] - 1, axis=1)
order = ['baseline', 'concat', 'concat-reg', 'film-final', 'leap', 'hybrid-adaln', 'hybrid-leap',
         'baseline-pr3p5', 'concat-pr3p5']
df['model'] = pd.Categorical(df.model, [m for m in order if m in set(df.model)] +
                             sorted(set(df.model) - set(order)))
df = df.sort_values(['checkpoint', 'model'], ascending=[False, True]).reset_index(drop=True)
fmt = {'E (meV)': '{:.2f}', 'F (meV/Å)': '{:.2f}', 'Δ E': '{:+.1%}', 'Δ F': '{:+.1%}'}
display(df.style.format({k: v for k, v in fmt.items() if k in df}).hide(axis='index'))
df.to_csv(f'{CKPT_DIR}/results_seed{SEED}{SUFFIX}.csv', index=False)